In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "Kudzai@chi9rus"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True, errors= 'ignore')

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

def build_query(filter_type):
    if filter_type == 'Water Rescue':
        return {
            'animal_type': 'Dog',
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156},
            '$or': [
                {'breed': {'$regex': 'Labrador Retriever', '$options': 'i'}},
                {'breed': {'$regex': 'Chesapeake Bay Retriever', '$options': 'i'}},
                {'breed': {'$regex': 'Newfoundland', '$options': 'i'}}
            ]
        }

    elif filter_type == 'Mountain or Wilderness Rescue':
        return {
            'animal_type': 'Dog',
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156},
            '$or': [
                {'breed': {'$regex': 'German Shepherd', '$options': 'i'}},
                {'breed': {'$regex': 'Alaskan Malamute', '$options': 'i'}},
                {'breed': {'$regex': 'Old English Sheepdog', '$options': 'i'}},
                {'breed': {'$regex': 'Siberian Husky', '$options': 'i'}},
                {'breed': {'$regex': 'Rottweiler', '$options': 'i'}}
            ]
        }

    elif filter_type == 'Disaster or Individual Tracking':
        return {
            'animal_type': 'Dog',
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300},
            '$or': [
                {'breed': {'$regex': 'Doberman Pinscher', '$options': 'i'}},
                {'breed': {'$regex': 'German Shepherd', '$options': 'i'}},
                {'breed': {'$regex': 'Golden Retriever', '$options': 'i'}},
                {'breed': {'$regex': 'Bloodhound', '$options': 'i'}},
                {'breed': {'$regex': 'Rottweiler', '$options': 'i'}}
            ]
        }

    else:
        return {}


app.layout = html.Div([
    html.Center(html.Img(
        src='data:image/png;base64,{}'.format(encoded_image.decode()),
        style={'height':  '200px'}
        
    )),
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('SNHU CS-340 Dashboard'))),
    html.Center(html.H3('Created by: Russell k Chimunya')),
    html.Hr(),
        
#FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.

  html.Div([
    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'Reset', 'value': 'Reset'},
            {'label': 'Water Rescue', 'value': 'Water Rescue'},
            {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain or Wilderness Rescue'},
            {'label': 'Disaster or Individual Tracking', 'value': 'Disaster or Individual Tracking'}
        ],
        value='Reset',
        inline=True
    )
]),
html.Hr(),
dash_table.DataTable(
    id='datatable-id',
    columns=[
        {"name": i, "id": i, "deletable": False, "selectable": True}
        for i in df.columns
    ],
    data=df.to_dict('records'),
    row_selectable='single',
    selected_rows=[0],
    page_size=10,
    sort_action='native',
    filter_action='native',
    column_selectable='single',
    style_table={'overflowX': 'auto'},
    style_cell={
        'textAlign': 'left',
        'minWidth': '100px',
        'width': '150px',
        'maxWidth': '200px',
        'whiteSpace': 'normal'
    },
    style_header={
        'backgroundColor': 'rgb(230, 230, 230)',
        'fontWeight': 'bold'
    }
),



#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here 

                        
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    query = build_query(filter_type)
    dff = pd.DataFrame.from_records(db.read(query))
    dff.drop(columns=['_id'], inplace=True, errors='ignore')
    return dff.to_dict('records')

## FIX ME Add code to filter interactive data table with MongoDB queries
#
#        
#        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
#        data=df.to_dict('records')
#       
#       
#        return (data,columns)

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    dff = pd.DataFrame.from_dict(viewData)

    if dff.empty:
        fig = px.pie(
            names=['No Data'],
            values=[1],
            title='Preferred Animals'
        )
    else:
        top_breeds = dff['breed'].value_counts().nlargest(10).reset_index()
        top_breeds.columns = ['breed', 'count']

        fig = px.pie(
            top_breeds,
            names='breed',
            values='count',
            title='Preferred Animals'
        )

    return [
        dcc.Graph(
            figure=fig
        )
    ]

    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        selected_columns = []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]



# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if viewData is None:
        dff = df
    else:
        dff = pd.DataFrame.from_dict(viewData)

    if dff.empty:
        return [html.Div('No matching animals found.')]

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    if row >= len(dff):
        row = 0

    animal_name = dff.iloc[row]['name']
    if pd.isna(animal_name) or animal_name == "":
        animal_name = "Unknown"

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[dff.iloc[row]['location_lat'], dff.iloc[row]['location_long']],
                    children=[
                        dl.Tooltip(str(dff.iloc[row]['breed'])),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(str(animal_name))
                        ])
                    ]
                )
            ]
        )
    ]

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://comrademayday-cricketeverest-3000.codio.io/proxy/8050/
